# Hyperview — SpectralGPT experiment

Uses **SpectralGPT** ([Hong et al., IEEE TPAMI 2024](https://doi.org/10.1109/TPAMI.2024.3362475)),
the first foundation model purpose-built for spectral remote sensing data.
Pretrained weights are downloaded from
[Zenodo 10533809](https://zenodo.org/records/10533809).

### Architecture
SpectralGPT treats spectral bands as a **temporal sequence** (video analogy):
each band is one grayscale 'frame'.  Spatial **8×8** and spectral **×4** patches
form 3D tokens fed into a ViT-Base/8 transformer.

```
12 selected bands  (B, 12, 128, 128)
  └─ unsqueeze  →  (B, 1, 12, 128, 128)          [band = time frame]
  └─ 3D PatchEmbed  8×8 spatial · 4 spectral
       → 16×16 spatial × 3 spectral = 768 tokens
  └─ ViT-Base  (12 blocks, 768-d, 12 heads)
  └─ global avg-pool  →  (B, 768)
  └─ + log(valid_pixels) scalar
  └─ MLP regressor  →  4 outputs  (P, K, Mg, pH)
```

### Band selection
SpectralGPT was pretrained on Sentinel-2 (12 bands). HyperView has 150 VNIR
bands (~430–900 nm).  We select **12 evenly-spaced bands** so the pretrained
spectral positional embeddings (3 groups of 4) align perfectly with the
checkpoint. The `BAND_SELECTION_STRATEGY` cell lets you switch to wavelength-
matched or custom selection.

### Key differences from DOFA notebook
| | DOFA | SpectralGPT |
|---|---|---|
| Input size | 224×224 | **128×128** |
| Bands fed to model | all 150 + wavelengths | **12 selected bands** |
| Spectral handling | wavelength-conditioned projection | 3D tokens (band = time frame) |
| Pretrained data | EO-1M (multi-sensor) | fMoW-Sentinel + BigEarthNet (Sentinel-2) |
| Batch size | 16 | **32** (smaller input → less VRAM) |


## 1 · Setup


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

for d in ['/content/train_data', '/content/test_data']:
    os.makedirs(d, exist_ok=True)

!tar -xzf /content/drive/MyDrive/train.tar.gz -C /content/train_data
!tar -xzf /content/drive/MyDrive/test.tar.gz  -C /content/test_data
!cp /content/drive/MyDrive/train_gt.csv    /content/train_gt.csv
!cp /content/drive/MyDrive/wavelengths.csv /content/wavelengths.csv
print('Data ready.')


### Clone SpectralGPT repository and download pretrained weights

We need the repo for `models_vit_tensor.py` (fine-tuning ViT) and
`util/pos_embed.py` (positional-embedding interpolation).
Weights are cached on Drive so they are not re-downloaded every session.


In [ ]:
# ── Clone SpectralGPT source code ──────────────────────────────────────
import os, sys

REPO_DIR = '/content/IEEE_TPAMI_SpectralGPT'
if not os.path.exists(REPO_DIR):
    !git clone --quiet https://github.com/danfenghong/IEEE_TPAMI_SpectralGPT.git {REPO_DIR}
    print('Repository cloned.')
else:
    print('Repository already present.')

# Make repo importable
sys.path.insert(0, REPO_DIR)
# Ensure util/ is a package
!touch {REPO_DIR}/util/__init__.py


In [ ]:
# ── Download SpectralGPT+ weights (cached on Drive) ────────────────────
# SpectralGPT+  = 200 epochs fMoW-Sentinel + 100 epochs BigEarthNet
# SpectralGPT   = 200 epochs fMoW-Sentinel only
# Both are ~1.2 GB — stored on Drive to avoid re-downloading each session.

DRIVE_WEIGHTS = '/content/drive/MyDrive/SpectralGPT+.pth'
LOCAL_WEIGHTS = '/content/SpectralGPT+.pth'

if os.path.exists(DRIVE_WEIGHTS):
    # Copy from Drive (fast)
    !cp '{DRIVE_WEIGHTS}' '{LOCAL_WEIGHTS}'
    print('Weights copied from Drive.')
elif not os.path.exists(LOCAL_WEIGHTS):
    # Download from Zenodo (slow, ~1.2 GB — run once)
    print('Downloading SpectralGPT+ weights from Zenodo (~1.2 GB) …')
    !wget -q --show-progress -O '{LOCAL_WEIGHTS}' \
        'https://zenodo.org/records/10533809/files/SpectralGPT%2B.pth'
    # Cache on Drive for future sessions
    !cp '{LOCAL_WEIGHTS}' '{DRIVE_WEIGHTS}'
    print('Weights saved to Drive for future sessions.')
else:
    print('Weights already present locally.')

WEIGHTS_PATH = LOCAL_WEIGHTS
print(f'Using weights: {WEIGHTS_PATH}')


In [ ]:
!pip install -q wandb joblib tqdm


In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    print('W&B key loaded from Colab Secrets.')
except Exception:
    # os.environ['WANDB_API_KEY'] = 'paste-your-key-here'
    print('WANDB_API_KEY not set — wandb.login() will prompt interactively.')


## 2 · Imports


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from glob import glob

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import wandb, joblib
from tqdm import tqdm

# SpectralGPT — fine-tuning ViT and positional-embedding utilities
from models_vit_tensor import VisionTransformer
from util.pos_embed import interpolate_pos_embed

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


## 3 · Data loading


In [ ]:
def load_data(directory: str):
    """Returns list of (img_tensor, mask) — img_tensor: (150, H, W), mask: (150, H, W) bool."""
    data = []
    all_files = np.array(sorted(
        glob(os.path.join(directory, '*.npz')),
        key=lambda x: int(os.path.basename(x).replace('.npz', ''))
    ))
    for f in all_files:
        with np.load(f) as npz:
            raw  = np.ma.MaskedArray(data=npz['data'], mask=npz['mask'])
            img  = torch.as_tensor(raw.data, dtype=torch.float32)
            mask = ~torch.as_tensor(raw.mask)   # True = valid
        data.append((img, mask))
    return data


def load_gt(file_path: str) -> np.ndarray:
    return pd.read_csv(file_path)[['P', 'K', 'Mg', 'pH']].values


X_all = load_data('/content/train_data/train')
y_all = load_gt('/content/train_gt.csv')
print(f'Patches: {len(X_all)}  |  Spectral bands: {X_all[0][0].shape[0]}')


In [ ]:
indices = np.arange(len(X_all))
X_tmp, X_test, y_tmp, y_test, idx_tmp, idx_test = train_test_split(
    X_all, y_all, indices, test_size=0.20, random_state=93
)
X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
    X_tmp, y_tmp, idx_tmp, test_size=0.25, random_state=93
)
print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')


## 4 · Band selection

SpectralGPT was pretrained on **12 Sentinel-2 bands** with `t_patch_size=4`,
producing exactly **3 spectral token positions**.  To reuse those pretrained
spectral positional embeddings without modification we must also feed
exactly 12 bands.

Three strategies are provided; choose one by setting `BAND_STRATEGY`:

| Strategy | Description |
|---|---|
| `'uniform'` | 12 evenly-spaced indices across 0–149 (default) |
| `'sentinel2_match'` | indices whose wavelengths are closest to Sentinel-2 VNIR bands |
| `'custom'` | set `CUSTOM_INDICES` manually |

> **Note:** HyperView covers only ~430–900 nm (VNIR).  Sentinel-2 SWIR bands
> (B11 = 1610 nm, B12 = 2190 nm) have no equivalent here, so
> `sentinel2_match` aligns with the 8 Sentinel-2 VNIR bands and fills the
> remaining 4 slots with evenly-spaced extras.


In [ ]:
# ── Load wavelengths ─────────────────────────────────────────────────────
wavelength_df = pd.read_csv('/content/wavelengths.csv')
WL_COL        = wavelength_df.columns[0]
wavelengths_nm = wavelength_df[WL_COL].values.astype(float)   # (150,) in nm

print(f'Spectral range: {wavelengths_nm.min():.1f} – {wavelengths_nm.max():.1f} nm  '
      f'({len(wavelengths_nm)} bands)')

# ── Band selection strategy ───────────────────────────────────────────────
BAND_STRATEGY  = 'uniform'    # 'uniform' | 'sentinel2_match' | 'custom'
N_SELECT       = 12           # must stay 12 to match pretrained spectral pos-embed
CUSTOM_INDICES = list(range(0, 150, 12))   # only used when BAND_STRATEGY='custom'

# Sentinel-2 VNIR band wavelengths (nm) — B10/B11/B12 are outside HyperView range
_S2_VNIR_NM = [490, 560, 665, 705, 740, 783, 842, 865]   # 8 bands

if BAND_STRATEGY == 'uniform':
    selected_indices = np.linspace(0, len(wavelengths_nm) - 1, N_SELECT, dtype=int)

elif BAND_STRATEGY == 'sentinel2_match':
    # Nearest-wavelength match for the 8 Sentinel-2 VNIR bands
    matched = [int(np.argmin(np.abs(wavelengths_nm - wl))) for wl in _S2_VNIR_NM]
    # Fill 4 extra slots with evenly-spaced bands not already selected
    remaining = [i for i in np.linspace(0, 149, 16, dtype=int) if i not in matched]
    extra      = remaining[:N_SELECT - len(matched)]
    selected_indices = np.array(sorted(matched + extra))

else:   # 'custom'
    selected_indices = np.array(CUSTOM_INDICES[:N_SELECT])

assert len(selected_indices) == N_SELECT, f'Expected {N_SELECT} bands, got {len(selected_indices)}'
selected_nm = wavelengths_nm[selected_indices]

print(f'\nSelected {N_SELECT} bands ({BAND_STRATEGY}):')
for i, (idx, wl) in enumerate(zip(selected_indices, selected_nm)):
    print(f'  Band {i+1:2d}: index {idx:3d}  ({wl:.1f} nm)')

# ── Visualise ─────────────────────────────────────────────────────────────
plt.figure(figsize=(10, 2))
plt.scatter(wavelengths_nm, np.zeros(150), s=8, c='lightblue', label='all 150 bands')
plt.scatter(selected_nm,    np.zeros(N_SELECT), s=60, c='red', zorder=5,
            label=f'{N_SELECT} selected')
plt.xlabel('Wavelength (nm)'); plt.yticks([])
plt.title(f'Band selection — strategy: {BAND_STRATEGY}')
plt.legend(); plt.tight_layout(); plt.show()


## 5 · Per-channel normalisation statistics

Computed over **valid pixels only** from the 12 selected bands of the training split.
Input size is **128×128** (SpectralGPT's native resolution, not 224×224).


In [ ]:
PAD_SIZE = (128, 128)   # SpectralGPT was pretrained at 128×128

def pad_to_size(x, mask, size=PAD_SIZE):
    _, h, w = x.shape
    th, tw  = size
    if h > th:
        sh = (h - th) // 2
        x, mask = x[:, sh:sh+th, :], mask[:, sh:sh+th, :]; h = th
    if w > tw:
        sw = (w - tw) // 2
        x, mask = x[:, :, sw:sw+tw], mask[:, :, sw:sw+tw]; w = tw
    ph, pw = th - h, tw - w
    pt, pl = ph // 2, pw // 2
    x    = F.pad(x,    (pl, pw-pl, pt, ph-pt), value=0)
    mask = F.pad(mask, (pl, pw-pl, pt, ph-pt), value=0)
    return x, mask


def calculate_global_stats(data_list, band_indices, size=PAD_SIZE):
    """Mean / std over valid pixels for the selected bands only."""
    n_ch   = len(band_indices)
    sums   = torch.zeros(n_ch)
    sums_sq= torch.zeros(n_ch)
    counts = torch.zeros(n_ch)
    for x, mask in tqdm(data_list, desc='Global stats'):
        xs   = x[band_indices]
        ms   = mask[band_indices]
        xp, mp = pad_to_size(xs, ms, size)
        valid  = mp[0].bool()
        vals   = xp[:, valid]                   # (n_ch, N_valid)
        sums   += vals.sum(dim=1)
        sums_sq+= (vals ** 2).sum(dim=1)
        counts += valid.sum().float()
    means = sums / counts
    stds  = (sums_sq / counts - means ** 2).sqrt().clamp(min=1e-6)
    return means, stds


print(f'Computing stats for {N_SELECT} selected bands at {PAD_SIZE} …')
global_means, global_stds = calculate_global_stats(X_train, selected_indices)
torch.save({'means': global_means, 'stds': global_stds,
            'band_indices': selected_indices.tolist()},
           'global_stats_spectralgpt.pt')
print('global_stats_spectralgpt.pt saved')


## 6 · Label scaling


In [ ]:
scaler_y = StandardScaler()
y_train_sc = scaler_y.fit_transform(y_train)
y_val_sc   = scaler_y.transform(y_val)
y_test_sc  = scaler_y.transform(y_test)
joblib.dump(scaler_y, 'scaler_labels_spectralgpt.pkl')
print('scaler_labels_spectralgpt.pkl saved')


## 7 · Dataset

Each item returns `(x, num_valid_pixels, y)` where `x` has shape
`(12, 128, 128)` — the 12 selected bands, padded/cropped and z-score
normalised.  The `VisionTransformer` forward adds the missing channel
dimension internally via `torch.unsqueeze(x, dim=1)`.


In [ ]:
class RandomRotate90:
    def __call__(self, x):
        k = torch.randint(0, 4, (1,)).item()
        return torch.rot90(x, k, dims=(-2, -1))


class RandomSpectralDrop:
    def __init__(self, drop_prob: float = 0.05):
        self.drop_prob = drop_prob
    def __call__(self, x):
        keep = torch.bernoulli(torch.ones(x.shape[0]) * (1 - self.drop_prob))
        return x * keep.view(-1, 1, 1)


class NPZDataset(Dataset):
    """Dataset for SpectralGPT: returns (12, 128, 128) tensors.

    band_indices : np.ndarray of shape (12,) — which of the 150 bands to keep
    """
    def __init__(self, tensor_list, labels=None, augment=True,
                 size=PAD_SIZE, global_means=None, global_stds=None,
                 band_indices=None):
        self.tensor_list  = tensor_list
        self.labels       = labels
        self.augment      = augment
        self.size         = size
        self.global_means = global_means
        self.global_stds  = global_stds
        self.band_indices = band_indices   # (12,) int array
        self.transform_aug = v2.Compose([
            v2.RandomResizedCrop(size=self.size, scale=(0.8, 1.0)),
            v2.RandomHorizontalFlip(p=0.5),
            v2.RandomVerticalFlip(p=0.5),
            RandomRotate90(),
            RandomSpectralDrop(drop_prob=0.05),
        ])

    def __len__(self):
        return len(self.tensor_list)

    def __getitem__(self, idx):
        x, mask = self.tensor_list[idx]       # (150, H, W)

        # ── select the 12 bands ──────────────────────────────────────────
        if self.band_indices is not None:
            x    = x[self.band_indices]        # (12, H, W)
            mask = mask[self.band_indices]     # (12, H, W)

        # ── pad / crop to 128×128 ────────────────────────────────────────
        x, mask = pad_to_size(x, mask, self.size)
        num_valid = mask[0].sum().float()

        # ── z-score normalisation (valid pixels only, then zero invalid) ─
        if self.global_means is not None:
            x = (x - self.global_means.view(-1,1,1)) / \
                (self.global_stds.view(-1,1,1) + 1e-6)
        x = x * mask.float()

        # ── augmentation ─────────────────────────────────────────────────
        if self.augment and self.labels is not None:
            x = self.transform_aug(x)
            x = x * mask.float()   # re-zero after spatial aug

        if self.labels is not None:
            return x, num_valid, torch.tensor(self.labels[idx], dtype=torch.float32)
        return x, num_valid


In [ ]:
BATCH_SIZE  = 32   # 128×128 input is smaller than 224×224 → fits fine on T4
NUM_WORKERS = 2

ds_kw = dict(global_means=global_means, global_stds=global_stds,
             band_indices=selected_indices)

train_ds = NPZDataset(X_train, y_train_sc, augment=True,  **ds_kw)
val_ds   = NPZDataset(X_val,   y_val_sc,   augment=False, **ds_kw)
test_ds  = NPZDataset(X_test,  y_test_sc,  augment=False, **ds_kw)

ldr_kw = dict(num_workers=NUM_WORKERS, pin_memory=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  **ldr_kw)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, **ldr_kw)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, **ldr_kw)

xb, nv, yb = next(iter(train_loader))
print('Batch shapes — x:', xb.shape, ' nv:', nv.shape, ' y:', yb.shape)
# Expected: x (32, 12, 128, 128)  nv (32,)  y (32, 4)


## 8 · SpectralGPT model

The `VisionTransformer` class from the SpectralGPT repo is used directly.
Its `forward` does `torch.unsqueeze(x, dim=1)` internally, so our
`(B, 12, 128, 128)` input becomes `(B, 1, 12, 128, 128)` — the standard
video format `(B, C, T, H, W)` with one colour channel per band.

After transformer processing, all patch tokens are averaged → `(B, 768)`
features which are then fed into the regression head.


In [ ]:
class SpectralGPTRegressor(nn.Module):
    """SpectralGPT ViT-Base backbone + MLP regression head.

    Args:
        n_outputs      : number of regression targets (4: P, K, Mg, pH)
        dropout        : dropout rate in the regression head
        pretrained_path: path to SpectralGPT .pth checkpoint (or None)
    """

    # Architecture must exactly match the pretrained checkpoint
    _BACKBONE_CFG = dict(
        num_frames    = 12,    # spectral bands fed to model
        t_patch_size  = 4,     # spectral token granularity: 12/4 = 3 spectral positions
        img_size      = 128,   # spatial resolution
        patch_size    = 8,     # spatial token size: 128/8 = 16 → 256 spatial positions
        in_chans      = 1,     # each band is a grayscale 'frame'
        embed_dim     = 768,
        depth         = 12,
        num_heads     = 12,
        mlp_ratio     = 4.0,
        drop_path_rate= 0.1,
        sep_pos_embed = True,  # separate spatial + temporal positional embeddings
        cls_embed     = False, # no CLS token — use global avg-pool instead
        num_classes   = 1,     # placeholder; replaced with Identity below
    )

    def __init__(
        self,
        n_outputs: int       = 4,
        dropout: float       = 0.3,
        pretrained_path: str = None,
    ):
        super().__init__()

        # ── backbone ────────────────────────────────────────────────────
        self.backbone = VisionTransformer(**self._BACKBONE_CFG)
        # Remove classification head — backbone now returns (B, 768) features
        self.backbone.head = nn.Identity()

        if pretrained_path is not None:
            self._load_pretrained(pretrained_path)
        else:
            print('[SpectralGPTRegressor] No pretrained weights — random init.')

        # ── regression head ─────────────────────────────────────────────
        # +1 for log(valid_pixel_count) patch-size feature
        embed_dim = self._BACKBONE_CFG['embed_dim']
        self.regressor = nn.Sequential(
            nn.Linear(embed_dim + 1, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(64, n_outputs),
        )

    def _load_pretrained(self, path: str):
        print(f'Loading SpectralGPT weights from {path} …')
        ckpt       = torch.load(path, map_location='cpu')
        ckpt_model = ckpt['model']          # MAE encoder weights
        state_dict = self.backbone.state_dict()

        # Drop keys with shape mismatch (patch_embed varies if band count differs;
        # head is replaced with Identity anyway)
        drop_keys = [
            'head.weight', 'head.bias',
            'patch_embed.0.proj.weight', 'patch_embed.1.proj.weight',
            'patch_embed.2.proj.weight', 'patch_embed.2.proj.bias',
        ]
        for k in drop_keys:
            if k in ckpt_model:
                if k not in state_dict or ckpt_model[k].shape != state_dict[k].shape:
                    print(f'  Skipping key: {k}')
                    del ckpt_model[k]

        # Interpolate positional embeddings if spatial/spectral size changed
        interpolate_pos_embed(self.backbone, ckpt_model)

        msg = self.backbone.load_state_dict(ckpt_model, strict=False)
        print(f'  Loaded.  Missing: {len(msg.missing_keys)}  '
              f'Unexpected: {len(msg.unexpected_keys)}')
        if msg.missing_keys:
            print(f'  Missing (first 5): {msg.missing_keys[:5]}')

    def forward(self, x: torch.Tensor, num_valid_pixels: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x                : (B, 12, 128, 128) — selected & normalised bands
                               VisionTransformer.forward adds channel dim internally
            num_valid_pixels : (B,) count of valid (non-padded) pixels
        Returns:
            (B, n_outputs)
        """
        features  = self.backbone(x)                                        # (B, 768)
        size_feat = torch.log1p(num_valid_pixels.float()) / 10.0            # (B,)
        features  = torch.cat([features, size_feat.unsqueeze(1)], dim=1)   # (B, 769)
        return self.regressor(features)


## 9 · Sanity-check: overfit on a single batch

Loss should reach < 0.01 within ~150 epochs.  This verifies that the
band selection, 3D patch embedding, and regression head are all wired up
correctly.


In [ ]:
model_test = SpectralGPTRegressor(
    pretrained_path = WEIGHTS_PATH,
    dropout         = 0.0,   # disable dropout for overfit test
).to(device)

x_s, nv_s, y_s = xb.to(device), nv.to(device), yb.to(device)

opt_s = optim.AdamW(model_test.parameters(), lr=1e-4, weight_decay=0)
crit  = nn.MSELoss()

print(f'Overfitting on batch of {x_s.shape[0]} samples for 150 epochs …')
for ep in range(150):
    model_test.train()
    opt_s.zero_grad()
    loss = crit(model_test(x_s, nv_s), y_s)
    loss.backward()
    opt_s.step()
    if (ep + 1) % 50 == 0 or ep == 0:
        print(f'  Epoch {ep+1:3d}/150   loss={loss.item():.6f}')

del model_test
torch.cuda.empty_cache()


## 10 · Training loop


In [ ]:
def train_with_early_stopping(
    model, train_loader, val_loader, optimizer, scheduler, criterion, device,
    epochs=90, warmup_epochs=10, patience=13,
    save_path='best_spectralgpt_model.pth', max_grad_norm=1.0, wandb_run=None
):
    """Linear LR warm-up · ReduceLROnPlateau · grad clipping · early stopping."""
    initial_lrs   = [pg['lr'] for pg in optimizer.param_groups]
    best_val_loss = float('inf')
    ctr           = 0

    for epoch in range(epochs):
        if epoch < warmup_epochs:
            wf = (epoch + 1) / float(warmup_epochs)
            for i, pg in enumerate(optimizer.param_groups):
                pg['lr'] = initial_lrs[i] * wf

        # ── train ──
        model.train()
        t_sum, t_n = 0.0, 0
        loop = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}', leave=False)
        for xb, nv, yb in loop:
            xb, nv, yb = xb.to(device), nv.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb, nv), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()
            t_sum += loss.item() * xb.size(0); t_n += xb.size(0)
            loop.set_postfix(loss=f'{loss.item():.4f}')
        train_loss = t_sum / max(1, t_n)

        # ── validate ──
        model.eval()
        v_sum, v_n = 0.0, 0
        with torch.no_grad():
            for xv, nv, yv in val_loader:
                xv, nv, yv = xv.to(device), nv.to(device), yv.to(device)
                v_sum += criterion(model(xv, nv), yv).item() * xv.size(0)
                v_n   += xv.size(0)
        val_loss = v_sum / max(1, v_n)

        if epoch >= warmup_epochs:
            scheduler.step(val_loss)

        lrs = [pg['lr'] for pg in optimizer.param_groups]
        print(f'Epoch {epoch+1:3d}/{epochs}  '
              f'Train: {train_loss:.4f}  Val: {val_loss:.4f}  '
              f'LR backbone: {lrs[0]:.2e}  LR head: {lrs[1]:.2e}')

        if wandb_run:
            wandb_run.log({'train_loss': train_loss, 'val_loss': val_loss,
                           'epoch': epoch + 1,
                           'lr_backbone': lrs[0], 'lr_head': lrs[1]})

        if val_loss < best_val_loss:
            best_val_loss = val_loss; ctr = 0
            torch.save(model.state_dict(), save_path)
            print(f'  Saved best model (val_loss={best_val_loss:.4f})')
        else:
            ctr += 1
            if ctr >= patience:
                print(f'Early stopping at epoch {epoch+1} (best: {best_val_loss:.4f})')
                break

    model.load_state_dict(torch.load(save_path, map_location=device))
    print(f'Best model loaded (val loss: {best_val_loss:.4f})')
    return model


## 11 · Build model, optimiser, and run training


In [ ]:
EPOCHS        = 90
WARMUP_EPOCHS = 10
PATIENCE      = 13
WEIGHT_DECAY  = 1e-4
LR_BACKBONE   = 1e-5   # pretrained backbone — fine-tune gently
LR_HEAD       = 1e-4   # randomly initialised head — train faster

model = SpectralGPTRegressor(
    pretrained_path = WEIGHTS_PATH,
    dropout         = 0.3,
).to(device)

optimizer = optim.AdamW([
    {'params': model.backbone.parameters(),  'lr': LR_BACKBONE},
    {'params': model.regressor.parameters(), 'lr': LR_HEAD},
], weight_decay=WEIGHT_DECAY)

criterion = nn.MSELoss()

scheduler = ReduceLROnPlateau(
    optimizer, mode='min', factor=0.3, patience=6, min_lr=1e-6
)

wandb.init(
    project = 'hyperview_challenge',
    name    = f'spectralgpt_base_{BAND_STRATEGY}12bands',
    config  = {
        'model':           'SpectralGPT-Base',
        'spectral_mode':   f'{BAND_STRATEGY}_12_bands',
        'selected_bands_nm': selected_nm.tolist(),
        'pretrained':      True,
        'weights':         'SpectralGPT+',
        'pad_size':        PAD_SIZE,
        'lr_backbone':     LR_BACKBONE,
        'lr_head':         LR_HEAD,
        'weight_decay':    WEIGHT_DECAY,
        'batch_size':      BATCH_SIZE,
        'epochs':          EPOCHS,
        'warmup_epochs':   WARMUP_EPOCHS,
        'patience':        PATIENCE,
        'criterion':       'MSELoss',
        'scheduler':       'ReduceLROnPlateau(factor=0.3, patience=6)',
    }
)

train_with_early_stopping(
    model         = model,
    train_loader  = train_loader,
    val_loader    = val_loader,
    optimizer     = optimizer,
    scheduler     = scheduler,
    criterion     = criterion,
    device        = device,
    epochs        = EPOCHS,
    warmup_epochs = WARMUP_EPOCHS,
    patience      = PATIENCE,
    save_path     = 'best_spectralgpt_model.pth',
    max_grad_norm = 1.0,
    wandb_run     = wandb.run,
)
wandb.finish()


## 12 · Evaluation vs. baseline


In [ ]:
class BaselineRegressor:
    def fit(self, X, y):
        self.mean = np.mean(y, axis=0); self.n = y.shape[1]; return self
    def predict(self, X):
        return np.full((len(X), self.n), self.mean)

class SpectralCurveFiltering:
    def __call__(self, x): return x.mean(axis=(1, 2))


def evaluate_dl_model(model, loader, scaler_y, device):
    model.eval()
    preds_sc, true_sc = [], []
    with torch.no_grad():
        for xb, nv, yb in loader:
            preds_sc.append(model(xb.to(device), nv.to(device)).cpu().numpy())
            true_sc.append(yb.numpy())
    y_pred = scaler_y.inverse_transform(np.vstack(preds_sc))
    y_true = scaler_y.inverse_transform(np.vstack(true_sc))
    mse    = np.mean((y_true - y_pred) ** 2, axis=0)
    return y_pred, y_true, mse


def calculate_and_print_results(model_mse, baseline_mse, y_true, y_pred, bl_pred):
    score  = np.mean(model_mse / baseline_mse)
    names  = ['P', 'K', 'Mg', 'pH']
    print('Per-target comparison:')
    for i, n in enumerate(names):
        print(f'  {n}: Model={model_mse[i]:.4f}  '
              f'Baseline={baseline_mse[i]:.4f}  '
              f'Norm={model_mse[i]/baseline_mse[i]:.4f}')
    print(f'\nChallenge score (lower is better): {score:.4f}')
    plt.figure(figsize=(15, 5))
    for i, n in enumerate(names):
        plt.subplot(1, 4, i + 1)
        plt.scatter(y_true[:,i], y_pred[:,i],  alpha=0.7, label='SpectralGPT')
        plt.scatter(y_true[:,i], bl_pred[:,i], alpha=0.7, label='Baseline', marker='x')
        lo = min(y_true[:,i].min(), y_pred[:,i].min())
        hi = max(y_true[:,i].max(), y_pred[:,i].max())
        plt.plot([lo, hi], [lo, hi], 'r--')
        plt.title(n); plt.xlabel('True'); plt.ylabel('Predicted')
        plt.legend(); plt.grid(True)
    plt.tight_layout(); plt.show()
    return score


In [ ]:
model.load_state_dict(torch.load('best_spectralgpt_model.pth', map_location=device))
dl_preds, y_test_true, dl_mse = evaluate_dl_model(model, test_loader, scaler_y, device)

filtering   = SpectralCurveFiltering()
X_test_raw  = [X_all[i][0] for i in idx_test]
X_test_filt = np.array([filtering(c.numpy()) for c in X_test_raw])

combined_idx = idx_train.tolist() + idx_val.tolist()
X_tv_filt    = np.array([filtering(X_all[i][0].numpy()) for i in combined_idx])

baseline = BaselineRegressor().fit(X_tv_filt, y_all[combined_idx])
bl_preds = baseline.predict(X_test_filt)
bl_mse   = np.mean((y_test_true - bl_preds) ** 2, axis=0)

score = calculate_and_print_results(dl_mse, bl_mse, y_test_true, dl_preds, bl_preds)


In [ ]:
names  = ['P', 'K', 'Mg', 'pH']
x_pos  = np.arange(len(names)); width = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
r1 = ax.bar(x_pos - width/2, dl_mse, width, label='SpectralGPT')
r2 = ax.bar(x_pos + width/2, bl_mse, width, label='Baseline')
for r in [r1, r2]:
    for rect in r:
        h = rect.get_height()
        ax.annotate(f'{h:.2f}',
                    xy=(rect.get_x() + rect.get_width()/2, h),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=9)
ax.set_xticks(x_pos); ax.set_xticklabels(names)
ax.set_ylabel('MSE'); ax.set_title('SpectralGPT vs. Baseline MSE per target')
ax.legend(); fig.tight_layout(); plt.show()
print(f'Challenge score: {score:.4f}')


## 13 · Submission inference


In [ ]:
import gc
del train_ds, val_ds, test_ds, train_loader, val_loader, test_loader
gc.collect(); torch.cuda.empty_cache()


In [ ]:
# ── Reload artefacts ──────────────────────────────────────────────────
saved       = torch.load('global_stats_spectralgpt.pt')
global_means= saved['means']
global_stds = saved['stds']
band_idx    = np.array(saved['band_indices'])
scaler_y    = joblib.load('scaler_labels_spectralgpt.pkl')

# ── Submission dataset ────────────────────────────────────────────────
X_sub    = load_data('/content/test_data/test')
sub_ds   = NPZDataset(X_sub, augment=False,
                      global_means=global_means, global_stds=global_stds,
                      band_indices=band_idx)
sub_loader = DataLoader(sub_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)

# ── Reload best model ─────────────────────────────────────────────────
model = SpectralGPTRegressor(pretrained_path=None).to(device)
model.load_state_dict(torch.load('best_spectralgpt_model.pth', map_location=device))
model.eval()

# ── Inference ─────────────────────────────────────────────────────────
preds_sc = []
with torch.no_grad():
    for xb, nv in sub_loader:
        preds_sc.append(model(xb.to(device), nv.to(device)).cpu().numpy())

import pandas as pd
y_pred = scaler_y.inverse_transform(np.vstack(preds_sc))
pd.DataFrame(y_pred, columns=['P', 'K', 'Mg', 'pH']).to_csv(
    'submission_spectralgpt.csv', index_label='sample_index'
)
print('submission_spectralgpt.csv saved')
